### Context Compression

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [2]:
load_dotenv()

True

Load the vector store, retriever, and reranker

In [3]:
# Load domain-tagged store
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Base retriever — pull 20 candidates
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 20})

# FlashRank reranker — keep the top 5
reranker = FlashrankRerank(model='ms-marco-MiniLM-L-12-v2', top_n=5)

# Reranking retriever
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

print(f'✅ Reranked retriever ready')
print(f'   Store has {vectorstore._collection.count()} chunks')
print(f'   Flow: 20 candidates -> rerank -> top 5')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Reranked retriever ready
   Store has 80 chunks
   Flow: 20 candidates -> rerank -> top 5


Build the compressor and the compressed retriever

In [4]:
# LLM used for extraction
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

extraction_prompt = PromptTemplate.from_template(
    "Extract from the passage only the parts that help answer the question.\n"
    "\n"
    "IMPORTANT RULES:\n"
    "1. Keep FULL sentences — never split a sentence in half.\n"
    "2. ALWAYS keep sentences with statistics, percentages, or numbers.\n"
    "3. Keep the introductory sentence of any list or table.\n"
    "4. If nothing is relevant, return an empty string.\n"
    "\n"
    "Passage:\n{context}\n\n"
    "Question: {question}\n\n"
    "Relevant text:"
)

compressor = LLMChainExtractor.from_llm(llm, prompt=extraction_prompt)



# Compressor: for each document, keep only sentences relevant to the query
compressor = LLMChainExtractor.from_llm(llm, prompt=extraction_prompt)

# Wrap the reranking retriever with the compressor
# Order: base retrieve (20) → rerank (top 5) → compress (extract relevant sentences)
compressed_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=reranking_retriever
)

print('Compressed retriever with custom prompt ready')
print('   Flow: 20 candidates → rerank (5) → compress (relevant sentences only)')


Compressed retriever with custom prompt ready
   Flow: 20 candidates → rerank (5) → compress (relevant sentences only)


Compare plain, reranked, and reranked + compressed

In [5]:
query = 'What percentage of deaths in Nigeria are caused by malaria?'

# --- 1. Base retrieval (top 5 by similarity, no reranking) ---
docs_base = base_retriever.invoke(query)

print('🔵 BASE RETRIEVAL')
print('=' * 70)
total_chars_base = 0

for i, d in enumerate(docs_base):
    total_chars_base += len(d.page_content)
    print(f'[{i}] {len(d.page_content)} chars')
    print('     ' + d.page_content[:200].replace('\n', ' '))
    print()
    
print(f'Total chars (base): {total_chars_base}')

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🔵 BASE RETRIEVAL
[0] 896 chars
     indirect cost of illness have also continued to           pauperize most of the citizens and this affect          economic growth as well as development. The sooner  governments begin to improve inves

[1] 775 chars
      The top causes of death in Nigeria are; malaria,  lower respiratory infections, HIV/AIDS,       diarrheal diseases, road injuries, protein -energy  malnutrition, cancer, meningitis, stroke and  tube

[2] 889 chars
     infectious diseases, sewage disposal, health insurance,  water supply, air pollution, noise pollution, environmen- tal radiation, housing, solid waste disposal, disaster  management, control of vector

[3] 897 chars
     health care services, brain drain, and irrational           appointment of health workers among others. A new  global burden has revealed that malaria and HIV are  still leading cause of death in Nige

[4] 850 chars
      Malaria (20%)   Lower Respiratory Infection (19%)   HIV/AIDS (9%)   Diarrhe

In [6]:
# --- 2. Reranked (top 5 after FlashRank) ---
docs_plain = reranking_retriever.invoke(query)

print('🟡 RERANKED ONLY (top 5 after FlashRank)')
print('=' * 70)

total_chars_plain = 0
for i, d in enumerate(docs_plain):
    total_chars_plain += len(d.page_content)
    print(f'[{i}] {len(d.page_content)} chars')
    print('     ' + d.page_content[:200].replace('\n', ' '))
    print()
    
print(f'Total chars after reranking: {total_chars_plain}')

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🟡 RERANKED ONLY (top 5 after FlashRank)
[0] 775 chars
      The top causes of death in Nigeria are; malaria,  lower respiratory infections, HIV/AIDS,       diarrheal diseases, road injuries, protein -energy  malnutrition, cancer, meningitis, stroke and  tube

[1] 897 chars
     health care services, brain drain, and irrational           appointment of health workers among others. A new  global burden has revealed that malaria and HIV are  still leading cause of death in Nige

[2] 889 chars
     infectious diseases, sewage disposal, health insurance,  water supply, air pollution, noise pollution, environmen- tal radiation, housing, solid waste disposal, disaster  management, control of vector

[3] 883 chars
     disease listed Nigeria and other developing countries as  the worst hit with deaths from non -communicable         diseases.5 These diseases with a rising burden in Nigeria  include cardiovascular dis

[4] 896 chars
     indirect cost of illness have also continued to          

In [7]:
# --- 3. Reranked + compressed ---
docs_compressed = compressed_retriever.invoke(query)

print('🟢 RERANKED + COMPRESSED')
print('=' * 70)
total_chars_compressed = 0

for i, d in enumerate(docs_compressed):
    total_chars_compressed += len(d.page_content)
    print(f'[{i}] {len(d.page_content)} chars')
    print('    ' + d.page_content[:200].replace('\n', ' '))
    print()
print(f'Total chars (compressed): {total_chars_compressed}')

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


🟢 RERANKED + COMPRESSED
[0] 154 chars
    Malaria remains the foremost killer disease in Nigeria. It accounts for over 25% of under 5 mortality, 30% childhood mortality and 11% maternal mortality.

[1] 166 chars
    A new global burden has revealed that malaria and HIV are still leading cause of death in Nigeria killing more than 190 thousand and 130 thousand people respectively.

[2] 72 chars
    The top 10 causes of death in Nigeria are as follows:3    Malaria (20%)

[3] 82 chars
    According to the 2011 World Health Statistics, malaria mortality rate for Nigeria.

[4] 319 chars
    Malaria remains the foremost killer disease in Nigeria. It has the highest burden of disease in Nigeria with an estimated 300,000 children dying of malaria each year. It accounts for over 25% of infan

Total chars (compressed): 793


In [8]:
# --- Side-by-side summary ---
print('\n' + '=' * 70)
print('📊 SUMMARY')
print('=' * 70)
print(f'Base retrieval:        {total_chars_base:>6} chars   (5 chunks)')
print(f'Reranked:              {total_chars_plain:>6} chars   (5 chunks)')
print(f'Reranked + compressed: {total_chars_compressed:>6} chars   (variable chunks)')

if total_chars_base:
    saved_vs_base = 100 * (1 - total_chars_compressed / total_chars_base)
    print(f'\n📉 Reduction vs base: {saved_vs_base:.1f}%')



📊 SUMMARY
Base retrieval:         16968 chars   (5 chunks)
Reranked:                4340 chars   (5 chunks)
Reranked + compressed:    793 chars   (variable chunks)

📉 Reduction vs base: 95.3%


Wire compression into the conversational ask() function

In [9]:
# Database for persistent history
DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')
DB_URL = 'sqlite:///' + DB_PATH.as_posix()

# Prompts
rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You rewrite a user\'s latest question into a standalone question using the chat history.\n'
     'Rules:\n'
     '1. If the latest question refers to something from the history (like "the first one", '
     '"that", "it", "the second option"), REPLACE that reference with the actual item from the history.\n'
     '2. Do NOT answer the question. Only rewrite it.\n'
     '3. If the question is already standalone, return it unchanged.\n'
     'Example:\n'
     '- History lists "Malaria" first, then "HIV/AIDS".\n'
     '  User: "Tell me more about the first one."\n'
     '  Rewritten: "Tell me more about malaria."'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. '
               'Answer the question using only the provided context. '
               'If the context contains a list, ranking, or enumeration that is relevant to the question, '
               'include the FULL list in your answer. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}'),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()

print('Prompts and chains ready')

Prompts and chains ready


Define the compressed conversational ask()

In [10]:
# Database for persistent history
DB_PATH = Path(r'C:\Users\USER\rag_course\chat_history.db')
DB_URL = 'sqlite:///' + DB_PATH.as_posix()

def get_history(session_id):
    '''Return a SQLite-backed chat history for this session'''
    
    # Build the history object with the current session's ID
    if not session_id or not isinstance(session_id, str):
        raise ValueError('session_id must be a non-empty string')
    
    # Sqlite chat history
    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DB_URL,
    )
    
    return history


def ask(question, session_id):
    '''Conversational RAG with reranking + compression + persistent history.'''
    
    # Load from SQLite
    history = get_history(session_id)
    
    # List of messages
    history_messages = history.messages
    
    # 1. Rewrite if history exists
    if history_messages:
        rewritten = rewrite_chain.invoke({
            'chat_history': history_messages,
            'input': question,
        })
        
    else:
        rewritten = question
        
    # 2. Retrieve + rerank + compress  ← one call, three stages
    docs = compressed_retriever.invoke(rewritten)
    context = '\n\n'.join(d.page_content for d in docs)
    
    # 3. Generate the answer
    answer = qa_chain.invoke({
        'context': context,
        'input': rewritten,
    })
    
    # 4. Persist both messages
    history.add_user_message(question)
    history.add_ai_message(answer)
    
    return answer

print('Compressed conversational RAG ready')

Compressed conversational RAG ready


Test the compressed conversational pipeline — Turn 1

In [11]:
session_id = 'compress-test-1'                                  # Fresh session

# Turn 1 — standalone question
q1 = 'What percentage of deaths in Nigeria are caused by malaria?'
a1 = ask(q1, session_id)

print('👤 Q1:', q1)
print('🤖 A1:', a1)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q1: What percentage of deaths in Nigeria are caused by malaria?
🤖 A1: Malaria accounts for 20% of deaths in Nigeria.
----------------------------------------------------------------------


In [12]:
query = 'What percentage of deaths in Nigeria are caused by malaria?'
docs = compressed_retriever.invoke(query)

print(f'{len(docs)} compressed chunks:\n')
for i, d in enumerate(docs):
    print(f'--- Chunk {i} ({len(d.page_content)} chars) ---')
    print(d.page_content)
    print()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


5 compressed chunks:

--- Chunk 0 (154 chars) ---
Malaria remains the foremost killer disease in Nigeria. It accounts for over 25% of under 5 mortality, 30% childhood mortality and 11% maternal mortality.

--- Chunk 1 (168 chars) ---
The new global burden has revealed that malaria and HIV are still leading cause of death in Nigeria killing more than 190 thousand and 130 thousand people respectively.

--- Chunk 2 (72 chars) ---
The top 10 causes of death in Nigeria are as follows:3 

 Malaria (20%)

--- Chunk 3 (82 chars) ---
According to the 2011 World Health Statistics, malaria mortality rate for Nigeria.

--- Chunk 4 (152 chars) ---
It accounts for over 25% of infant mortality (children under aged one), 30% of childhood mortality (children under five), and 11% of maternal mortality.



In [13]:
# Fresh session so we test from scratch
session_id = 'compressed-test-2'

q1 = 'What percentage of deaths in Nigeria are caused by malaria?'
a1 = ask(q1, session_id)

print('👤 Q1:', q1)
print('🤖 A1:', a1)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q1: What percentage of deaths in Nigeria are caused by malaria?
🤖 A1: The percentage of deaths in Nigeria caused by HIV/AIDS is 9%.
----------------------------------------------------------------------


Multi‑turn test — vague follow‑up

In [14]:
# Turn 2 — vague follow-up in the same session
q2 = 'What about the second leading cause?'
a2 = ask(q2, session_id)

print('👤 Q2:', q2)
print('🤖 A2:', a2)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q2: What about the second leading cause?
🤖 A2: The percentage of deaths in Nigeria caused by HIV/AIDS is 9%.
----------------------------------------------------------------------
